In [1]:
# ==============================
# Federated Learning on Fashion-MNIST (Dirichlet Non-IID)
# Model: LeNet++ CNN
# Algorithm: FedAvg ONLY
# Tracks: Train/Val/Test Loss & Accuracy + Round Time + Total Time
# ==============================

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import numpy as np
import random
import copy
import time
from collections import OrderedDict

# -------------------------
# Configuration
# -------------------------
NUM_CLIENTS = 100
CLIENT_FRAC = 0.3
ROUNDS = 60
LOCAL_EPOCHS = 3
BATCH_SIZE = 128
LR = 0.01
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
ALPHA = 0.1

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

print("Device:", DEVICE)

# -------------------------
# Dataset: Fashion-MNIST
# -------------------------
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.FashionMNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.FashionMNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

# -------------------------
# Clients split
# -------------------------
VAL_CLIENTS = list(range(10))
TRAIN_CLIENTS = list(range(10, NUM_CLIENTS))

# -------------------------
# Dirichlet Non-IID
# -------------------------
def dirichlet_partition(dataset, clients, alpha=0.1):

    num_classes = 10

    labels = np.array(dataset.targets)

    idxs = np.arange(len(labels))

    client_dict = {i: [] for i in clients}

    class_indices = [
        idxs[labels == i]
        for i in range(num_classes)
    ]

    for c_idx in class_indices:

        np.random.shuffle(c_idx)

        proportions = np.random.dirichlet(
            alpha * np.ones(len(clients))
        )

        proportions = (
            np.cumsum(proportions) * len(c_idx)
        ).astype(int)

        proportions = np.concatenate(([0], proportions))

        for i, client in enumerate(clients):

            client_dict[client] += list(
                c_idx[proportions[i]:proportions[i+1]]
            )

    for client in clients:
        np.random.shuffle(client_dict[client])

    return client_dict

client_data = dirichlet_partition(
    train_dataset,
    list(range(NUM_CLIENTS)),
    ALPHA
)

# -------------------------
# Model
# -------------------------
class FMNIST_CNN(nn.Module):

    def __init__(self):

        super().__init__()

        self.conv1 = nn.Conv2d(
            1,
            32,
            3,
            padding=1
        )

        self.conv2 = nn.Conv2d(
            32,
            64,
            3,
            padding=1
        )

        self.pool = nn.MaxPool2d(2)

        self.fc1 = nn.Linear(
            64 * 7 * 7,
            256
        )

        self.fc2 = nn.Linear(
            256,
            10
        )

    def forward(self, x):

        x = self.pool(
            F.relu(self.conv1(x))
        )

        x = self.pool(
            F.relu(self.conv2(x))
        )

        x = x.view(x.size(0), -1)

        x = F.relu(self.fc1(x))

        return self.fc2(x)

# -------------------------
# Evaluation
# -------------------------
def evaluate(model, loader):

    model.eval()

    correct, total, loss_val = 0, 0, 0.0

    with torch.no_grad():

        for x, y in loader:

            x, y = x.to(DEVICE), y.to(DEVICE)

            out = model(x)

            loss_val += (
                F.cross_entropy(out, y).item()
                * y.size(0)
            )

            pred = out.argmax(dim=1)

            correct += (pred == y).sum().item()

            total += y.size(0)

    return (
        100 * correct / total,
        loss_val / total
    )

# -------------------------
# Federated Training (FedAvg ONLY)
# -------------------------
global_model = FMNIST_CNN().to(DEVICE)

train_acc, val_acc, test_acc = [], [], []

loss_curve = []
train_loss_curve = []
val_loss_curve = []
test_loss_curve = []

round_time, total_time = [], []

start_total = time.time()

for rnd in range(ROUNDS):

    start_round = time.time()

    selected_clients = random.sample(
        TRAIN_CLIENTS,
        int(NUM_CLIENTS * CLIENT_FRAC)
    )

    local_weights = []
    local_sizes = []
    local_losses = []

    # -------------------------
    # Local training
    # -------------------------
    for cid in selected_clients:

        local_model = copy.deepcopy(global_model)

        local_model.train()

        optimizer = optim.SGD(
            local_model.parameters(),
            lr=LR,
            momentum=0.9,
            weight_decay=5e-4
        )

        loader = DataLoader(
            Subset(train_dataset, client_data[cid]),
            batch_size=BATCH_SIZE,
            shuffle=True
        )

        for _ in range(LOCAL_EPOCHS):

            for x, y in loader:

                x, y = x.to(DEVICE), y.to(DEVICE)

                optimizer.zero_grad()

                loss = F.cross_entropy(
                    local_model(x),
                    y
                )

                loss.backward()

                optimizer.step()

        local_losses.append(loss.item())

        local_weights.append(
            copy.deepcopy(local_model.state_dict())
        )

        local_sizes.append(
            len(client_data[cid])
        )

    # -------------------------
    # FedAvg aggregation
    # -------------------------
    total = sum(local_sizes)

    new_weights = OrderedDict()

    for k in global_model.state_dict().keys():

        new_weights[k] = sum(
            local_weights[i][k]
            * (local_sizes[i] / total)
            for i in range(len(local_weights))
        )

    global_model.load_state_dict(new_weights)

    # -------------------------
    # Metrics
    # -------------------------
    tr_acc, tr_loss = evaluate(
        global_model,
        DataLoader(
            torch.utils.data.ConcatDataset([
                Subset(train_dataset, client_data[c])
                for c in selected_clients
            ]),
            batch_size=BATCH_SIZE
        )
    )

    va_acc, va_loss = evaluate(
        global_model,
        DataLoader(
            torch.utils.data.ConcatDataset([
                Subset(train_dataset, client_data[c])
                for c in VAL_CLIENTS
            ]),
            batch_size=BATCH_SIZE
        )
    )

    te_acc, te_loss = evaluate(
        global_model,
        DataLoader(
            test_dataset,
            batch_size=BATCH_SIZE
        )
    )

    # logs
    loss_curve.append(np.mean(local_losses))

    train_loss_curve.append(tr_loss)
    val_loss_curve.append(va_loss)
    test_loss_curve.append(te_loss)

    train_acc.append(tr_acc)
    val_acc.append(va_acc)
    test_acc.append(te_acc)

    rt = time.time() - start_round

    round_time.append(rt)

    total_time.append(
        time.time() - start_total
    )

    print(
        f"Round {rnd+1:03d} | "
        f"LocalLoss {loss_curve[-1]:.4f} | "
        f"TrainLoss {tr_loss:.4f} | "
        f"ValLoss {va_loss:.4f} | "
        f"TestLoss {te_loss:.4f} | "
        f"Train {tr_acc:.4f} | "
        f"Val {va_acc:.4f} | "
        f"Test {te_acc:.4f} | "
        f"RoundTime {rt:.2f}s | "
        f"TotalTime {total_time[-1]:.2f}s"
    )

Device: cuda


100%|██████████| 26.4M/26.4M [00:02<00:00, 10.3MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 168kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.19MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 10.1MB/s]


Round 001 | LocalLoss 0.8663 | TrainLoss 1.7189 | ValLoss 2.1278 | TestLoss 2.0939 | Train 59.9875 | Val 40.7892 | Test 41.2400 | RoundTime 22.66s | TotalTime 22.66s
Round 002 | LocalLoss 0.6233 | TrainLoss 1.8305 | ValLoss 1.9930 | TestLoss 1.9952 | Train 28.7681 | Val 27.6446 | Test 21.1300 | RoundTime 19.18s | TotalTime 41.84s
Round 003 | LocalLoss 0.4686 | TrainLoss 1.4439 | ValLoss 1.5917 | TestLoss 1.5570 | Train 40.1603 | Val 35.0763 | Test 34.2600 | RoundTime 22.42s | TotalTime 64.26s
Round 004 | LocalLoss 0.4244 | TrainLoss 1.3627 | ValLoss 1.6699 | TestLoss 1.8399 | Train 44.9514 | Val 31.9535 | Test 31.5600 | RoundTime 15.67s | TotalTime 79.93s
Round 005 | LocalLoss 0.3845 | TrainLoss 1.2494 | ValLoss 1.5693 | TestLoss 1.4392 | Train 46.4239 | Val 27.7173 | Test 35.6200 | RoundTime 18.99s | TotalTime 98.92s
Round 006 | LocalLoss 0.4019 | TrainLoss 0.9858 | ValLoss 1.2092 | TestLoss 1.0470 | Train 63.8967 | Val 46.0179 | Test 58.2700 | RoundTime 22.64s | TotalTime 121.55s
Rou

In [2]:
# -------------------------
# Save metrics to CSV
# -------------------------
import csv

metrics = {
    "round": list(range(1, ROUNDS+1)),
    "train_acc": train_acc,
    "val_acc": val_acc,
    "test_acc": test_acc,
    "local_loss": loss_curve,
    "train_loss": train_loss_curve,
    "val_loss": val_loss_curve,
    "test_loss": test_loss_curve,
    "round_time": round_time,
    "total_time": total_time
}

with open("FMNIST_fedavg_federated_metrics_0.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(metrics.keys())
    writer.writerows(zip(*metrics.values()))

# Colab download
try:
    from google.colab import files
    files.download("FMNIST_fedavg_federated_metrics_0.csv")
except:
    print("CSV saved locally as FMNIST_fedavg_federated_metrics_0.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>